# Waste Transportation Network Optimization

## Business use case

Waste operators need to route material from generation centers through optional transfer depots to final landfills while respecting capacity and demand constraints. Small routing inefficiencies become expensive at hundreds of thousands of tons, so this is a natural network-optimization problem.

## Objective

The model minimizes total transportation cost across two generation centers, four transfer depots, and six landfills. It allows both direct center-to-landfill shipments and center-to-depot-to-landfill flows while enforcing supply, throughput, flow-conservation, and landfill-demand constraints.

## Result in context

The stored Gurobi solution is optimal with a total transportation cost of **$541,000** for the stated demand scenario.

## Conditions

There are six end landfills, each with a known demand for a waste material. After two years, two new landfills will be aggregated (C7 and C8), one each year. Landfill demand can be satisfied from a set of four transfer station depots, or directly from a set of two waste generating centers.  Each transfer depot can support a maximum volume of waste moving through it, and each waste generating center can generate a maximum amount of waste.  There are known costs associated with transporting the waste, from a center to a depot, from a depot to a landfill, or from a center directly to a landfill.

The waste network has two waste generating centers, in NewYork and NewJersey, which generate the waste.  Each has a maximum waste generating volume:

| Center | Waste (tons) |
| --- | --- |
| NewYork | 300,000 |
| NewJersey |  400,000 |

The waste can be shipped from a center to a set of four depots. Each depot has a maximum throughput.

| Depot | Throughput (tons) |
| --- | --- |
| Bronx | 140,000 |
| Brooklyn | 100,000 |
| Queens | 200,000 |
| StatenIsland | 80,000 |

The network has six landfills, each with a given maximum demand. After two years, two new landfills will be aggregated (C7 and C8), one each year.

| Landfill | Demand (tons) |
| --- | --- |
| C1 | 100,000 |
| C2 | 20,000 |
| C3 | 80,000 |
| C4 | 70,000 |
| C5 | 120,000 |
| C6 | 40,000 |
| C7 | 60,000 |
| C8 | 130,000 |

Transporation costs are given in the following table (in dollars per ton).  Columns are source cities and rows are destination cities.  

| To | NewYork | NewJersey | Bronx | Brooklyn | Queens | StatenIsland |
| --- | --- | --- | --- | --- | --- | --- |
| Depots |
| Bronx  | 0.7 |   - |
| Brooklyn | 0.7 | 0.5 |
| Queens     | 1.2 | 0.7 |
| StatenIsland     | 0.4 | 0.4 |
| Landfills |
| C1 | 1.2 | 2.2 |   - | 1.2 |   - |   - |
| C2 |   - |   - | 1.7 | 0.7 | 1.7 |   - |
| C3 | 1.7 |   - | 0.7 | 0.75 | 2.2 | 0.4 |
| C4 | 2.2 |   - | 1.7 | 1.2|   - | 1.7 |
| C5 |   - |   - |   - | 0.7 | 0.7 | 0.7 |
| C6 | 1.2 |   - | 1.2 |   - | 1.7 | 1.7 |
| C7 | 1.8 | 1.9 | 0.8 | 0.4 | 0.6 | 1.9 |
| C8 | 1.5 | 1.7 | 0.7 | 2.0 | 1.5 | 1.4 |


## Step 1 — Set up the optimization stack

The notebook installs and imports Gurobi and Pandas. Gurobi is used to formulate and solve the linear network-flow model.


In [ ]:
%pip install gurobipy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.9/14.9 MB 70.8 MB/s eta 0:00:00


In [ ]:
import pandas as pd

import gurobipy as gp
from gurobipy import GRB

# tested with Python 3.7.0 & Gurobi 9.0

## Step 2 — Encode capacities, demand, and lane costs

Supply limits are defined for New York and New Jersey, throughput limits for the four transfer depots, and demand for six landfills. A cost dictionary represents the permitted shipping lanes and their per-ton costs.

### Sets and Indices

$f \in \text{Centers}=\{\text{NewYork}, \text{NewJersey}\}$

$d \in \text{Depots}=\{\text{Bronx}, \text{Brooklyn}, \text{Queens}, \text{StatenIsland}\}$

$c \in \text{Landfills}=\{\text{C1}, \text{C2}, \text{C3}, \text{C4}, \text{C5}, \text{C6}\}$

$\text{Locations} = \text{Centers} \cup \text{Depots} \cup \text{Landfills}$

### Parameters

$\text{cost}_{s,t} \in \mathbb{R}^+$: Cost of shipping one ton from source $s$ to destination $t$.

$\text{supply}_f \in \mathbb{R}^+$: Maximum possible supply from center $f$ (in tons).

$\text{through}_d \in \mathbb{R}^+$: Maximum possible flow through depot $d$ (in tons).

$\text{demand}_c \in \mathbb{R}^+$: Demand for waste at landfill $c$ (in tons).


In [ ]:
# Create dictionaries to capture center supply limits, depot throughput limits, and landfill demand.

supply = dict({'NewYork': 300000,
               'NewJersey': 400000})

through = dict({'Bronx': 140000,
                'Brooklyn': 100000,
                'Queens': 200000,
                'StatenIsland': 80000})

demand = dict({'C1': 100000,
               'C2': 20000,
               'C3': 80000,
               'C4': 70000,
               'C5': 120000,
               'C6': 40000})

# Create a dictionary to capture shipping costs.

arcs, cost = gp.multidict({
    ('NewYork', 'Bronx'): 0.7,
    ('NewYork', 'Brooklyn'): 0.7,
    ('NewYork', 'Queens'): 1.2,
    ('NewYork', 'StatenIsland'): 0.4,
    ('NewYork', 'C1'): 1.2,
    ('NewYork', 'C3'): 1.7,
    ('NewYork', 'C4'): 2.2,
    ('NewYork', 'C6'): 1.2,
    ('NewJersey', 'Brooklyn'): 0.5,
    ('NewJersey', 'Queens'): 0.7,
    ('NewJersey', 'StatenIsland'): 0.4,
    ('NewJersey', 'C1'): 2.2,
    ('Bronx', 'C2'): 1.7,
    ('Bronx', 'C3'): 0.7,
    ('Bronx', 'C4'): 1.7,
    ('Bronx', 'C6'): 1.2,
    ('Brooklyn', 'C1'): 1.2,
    ('Brooklyn', 'C2'): 0.7,
    ('Brooklyn', 'C3'): 0.75,
    ('Brooklyn', 'C4'): 1.2,
    ('Brooklyn', 'C5'): 0.7,
    ('Queens', 'C2'): 1.7,
    ('Queens', 'C3'): 2.2,
    ('Queens', 'C5'): 0.7,
    ('Queens', 'C6'): 1.7,
    ('StatenIsland', 'C3'): 0.4,
    ('StatenIsland', 'C4'): 1.7,
    ('StatenIsland', 'C5'): 0.7,
    ('StatenIsland', 'C6'): 1.7
})

## Step 3 — Define flow decision variables and the objective

A nonnegative flow variable is created for every allowed arc. Transportation cost is attached directly to each variable, so minimizing the model objective minimizes total shipping cost across the network.

### Decision Variables

$\text{flow}_{s,t} \in \mathbb{N}^+$: Quantity of waste (in tons) that is shipped from source $s$ to destionation $t$.


### Objective Function

- **Cost**: Minimize total shipping costs.

\begin{equation}
\text{Minimize} \quad Z = \sum_{(s,t) \in \text{locations} \times \text{locations}}{\text{cost}_{s,t}*\text{flow}_{s,t}}
\end{equation}

In [ ]:
model = gp.Model('SupplyNetworkDesign')
flow = model.addVars(arcs, obj=cost, name="flow")

Restricted license - for non-production use only - expires 2027-11-29


## Step 4 — Add operational constraints

Center outflow cannot exceed available supply. Each landfill must receive exactly its demand. Transfer depots must conserve flow, and inbound depot flow cannot exceed throughput capacity.

- **Center output**: Flow of goods from a factory must respect maximum capacity.

\begin{equation}
\sum_{t \in \text{locations}}{\text{flow}_{f,t}} \leq \text{supply}_{f} \quad \forall f \in \text{Centers}
\end{equation}

- **Landfill demand**: Flow of goods must meet customer demand.

\begin{equation}
\sum_{s \in \text{locations}}{\text{flow}_{s,c}} = \text{demand}_{c} \quad \forall c \in \text{Landfills}
\end{equation}

- **Depot flow**: Flow into a depot equals flow out of the depot.

\begin{equation}
\sum_{s \in \text{locations}}{\text{flow}_{s,d}} =
\sum_{t \in \text{locations}}{\text{flow}_{d,t}}
\quad \forall d \in \text{Depots}
\end{equation}

- **Depot capacity**: Flow into a depot must respect depot capacity.

\begin{equation}
\sum_{s \in \text{locations}}{\text{flow}_{s,d}} \leq \text{through}_{d}
\quad \forall d \in \text{Depots}
\end{equation}


In [ ]:
# Center capacity limits

centers = supply.keys()
center_flow = model.addConstrs((gp.quicksum(flow.select(center, '*')) <= supply[center]
                                 for center in centers), name="center")

In [ ]:
# landfill demand

landfills = demand.keys()
landfill_flow = model.addConstrs((gp.quicksum(flow.select('*', landfill)) == demand[landfill]
                                  for landfill in landfills), name="landfill")

In [ ]:
# Depot flow conservation

depots = through.keys()
depot_flow = model.addConstrs((gp.quicksum(flow.select(depot, '*')) == gp.quicksum(flow.select('*', depot))
                               for depot in depots), name="depot")

In [ ]:
# Depot throughput

depot_capacity = model.addConstrs((gp.quicksum(flow.select('*', depot)) <= through[depot]
                                   for depot in depots), name="depot_capacity")

## Step 5 — Solve for the minimum-cost network

Gurobi solves the linear model to optimality. The stored run reaches an objective of **$541,000**.


In [ ]:
model.optimize()

Gurobi Optimizer version 13.0.3 build v13.0.3rc0 (linux64 - "Ubuntu 24.04.4 LTS")

CPU model: Intel(R) Xeon(R) CPU @ 2.20GHz, instruction set [SSE2|AVX|AVX2]
Thread count: 1 physical cores, 2 logical processors, using up to 2 threads

Optimize a model with 16 rows, 29 columns and 65 nonzeros (Min)
Model fingerprint: 0x68eb5bfa
Model has 29 linear objective coefficients
Coefficient statistics:
  Matrix range     [1e+00, 1e+00]
  Objective range  [4e-01, 2e+00]
  Bounds range     [0e+00, 0e+00]
  RHS range        [2e+04, 4e+05]

Presolve removed 1 rows and 0 columns
Presolve time: 0.02s
Presolved: 15 rows, 29 columns, 64 nonzeros

Iteration    Objective       Primal Inf.    Dual Inf.      Time
       0    3.8200000e+05   3.625000e+04   0.000000e+00      0s
       7    5.4100000e+05   0.000000e+00   0.000000e+00      0s

Solved in 7 iterations and 0.03 seconds (0.00 work units)
Optimal objective  5.410000000e+05


## Step 6 — Translate the optimizer output into a shipping plan

Only positive-flow arcs are retained in the final table. This makes the mathematical solution actionable by showing the exact lanes and tonnage required under the current cost and capacity assumptions.


In [ ]:
results = []
for arc in arcs:
    if flow[arc].x > 1e-6:
        results.append({"From": arc[0], "To": arc[1], "Flow": flow[arc].x})
product_flow = pd.DataFrame(results)
product_flow.index=[''] * len(product_flow)
product_flow

,From,To,Flow
,NewYork,C1,100000.0
,NewYork,C6,40000.0
,NewJersey,Brooklyn,100000.0
,NewJersey,Queens,110000.0
,NewJersey,StatenIsland,80000.0
,Brooklyn,C2,20000.0
,Brooklyn,C4,70000.0
,Brooklyn,C5,10000.0
,Queens,C5,110000.0
,StatenIsland,C3,80000.0


## Step 7: Optimize new network after one year

In [ ]:
# Update demand limit
demand['C7'] = 60000

# Update arcs and cost
cost[('NewYork', 'C7')] = 1.8
cost[('NewJersey', 'C7')] = 1.9
cost[('Bronx', 'C7')] = 0.8
cost[('Brooklyn', 'C7')] = 0.4
cost[('Queens', 'C7')] = 0.6
cost[('StatenIsland', 'C7')] = 1.9

arcs.append(('NewYork', 'C7'))
arcs.append(('NewJersey', 'C7'))
arcs.append(('Bronx', 'C7'))
arcs.append(('Brooklyn', 'C7'))
arcs.append(('Queens', 'C7'))
arcs.append(('StatenIsland', 'C7'))

model = gp.Model('SupplyNetworkDesign')
flow = model.addVars(arcs, obj=cost, name="flow")

# Center capacity limits
centers = supply.keys()
center_flow = model.addConstrs((gp.quicksum(flow.select(center, '*')) <= supply[center]
                                 for center in centers), name="center")

# landfill demand
landfills = demand.keys()
landfill_flow = model.addConstrs((gp.quicksum(flow.select('*', landfill)) == demand[landfill]
                                  for landfill in landfills), name="landfill")

# Depot flow conservation
depots = through.keys()
depot_flow = model.addConstrs((gp.quicksum(flow.select(depot, '*')) == gp.quicksum(flow.select('*', depot))
                               for depot in depots), name="depot")

# Depot throughput
depot_capacity = model.addConstrs((gp.quicksum(flow.select('*', depot)) <= through[depot]
                                   for depot in depots), name="depot_capacity")

model.optimize()

results = []
for arc in arcs:
    if flow[arc].x > 1e-6:
        results.append({"From": arc[0], "To": arc[1], "Flow": flow[arc].x})
product_flow = pd.DataFrame(results)
product_flow.index=[''] * len(product_flow)
product_flow

Gurobi Optimizer version 13.0.3 build v13.0.3rc0 (linux64 - "Ubuntu 24.04.4 LTS")

CPU model: Intel(R) Xeon(R) CPU @ 2.20GHz, instruction set [SSE2|AVX|AVX2]
Thread count: 1 physical cores, 2 logical processors, using up to 2 threads

Optimize a model with 17 rows, 35 columns and 77 nonzeros (Min)
Model fingerprint: 0xc7035754
Model has 35 linear objective coefficients
Coefficient statistics:
  Matrix range     [1e+00, 1e+00]
  Objective range  [4e-01, 2e+00]
  Bounds range     [0e+00, 0e+00]
  RHS range        [2e+04, 4e+05]

Presolve removed 1 rows and 0 columns
Presolve time: 0.02s
Presolved: 16 rows, 35 columns, 76 nonzeros

Iteration    Objective       Primal Inf.    Dual Inf.      Time
       0    4.0600000e+05   4.375000e+04   0.000000e+00      0s
      12    6.1700000e+05   0.000000e+00   0.000000e+00      0s

Solved in 12 iterations and 0.04 seconds (0.00 work units)
Optimal objective  6.170000000e+05


,From,To,Flow
,NewYork,C1,100000.0
,NewYork,C6,40000.0
,NewJersey,Brooklyn,100000.0
,NewJersey,Queens,170000.0
,NewJersey,StatenIsland,80000.0
,Brooklyn,C2,20000.0
,Brooklyn,C4,70000.0
,Queens,C5,120000.0
,StatenIsland,C3,80000.0
,Brooklyn,C7,10000.0


## Step 8: Optimize new network after two years

In [ ]:
demand['C8'] = 130000

# Update arcs and cost
cost[('NewYork', 'C8')] = 1.5
cost[('NewJersey', 'C8')] = 1.7
cost[('Bronx', 'C8')] = 0.7
cost[('Brooklyn', 'C8')] = 2.0
cost[('Queens', 'C8')] = 1.5
cost[('StatenIsland', 'C8')] = 1.4

arcs.append(('NewYork', 'C8'))
arcs.append(('NewJersey', 'C8'))
arcs.append(('Bronx', 'C8'))
arcs.append(('Brooklyn', 'C8'))
arcs.append(('Queens', 'C8'))
arcs.append(('StatenIsland', 'C8'))

model = gp.Model('SupplyNetworkDesign')
flow = model.addVars(arcs, obj=cost, name="flow")

# Center capacity limits
centers = supply.keys()
center_flow = model.addConstrs((gp.quicksum(flow.select(center, '*')) <= supply[center]
                                 for center in centers), name="center")

# landfill demand
landfills = demand.keys()
landfill_flow = model.addConstrs((gp.quicksum(flow.select('*', landfill)) == demand[landfill]
                                  for landfill in landfills), name="landfill")

# Depot flow conservation
depots = through.keys()
depot_flow = model.addConstrs((gp.quicksum(flow.select(depot, '*')) == gp.quicksum(flow.select('*', depot))
                               for depot in depots), name="depot")

# Depot throughput
depot_capacity = model.addConstrs((gp.quicksum(flow.select('*', depot)) <= through[depot]
                                   for depot in depots), name="depot_capacity")

model.optimize()

results = []
for arc in arcs:
    if flow[arc].x > 1e-6:
        results.append({"From": arc[0], "To": arc[1], "Flow": flow[arc].x})
product_flow = pd.DataFrame(results)
product_flow.index=[''] * len(product_flow)
product_flow

Gurobi Optimizer version 13.0.3 build v13.0.3rc0 (linux64 - "Ubuntu 24.04.4 LTS")

CPU model: Intel(R) Xeon(R) CPU @ 2.20GHz, instruction set [SSE2|AVX|AVX2]
Thread count: 1 physical cores, 2 logical processors, using up to 2 threads

Optimize a model with 18 rows, 41 columns and 89 nonzeros (Min)
Model fingerprint: 0xf71fa3ba
Model has 41 linear objective coefficients
Coefficient statistics:
  Matrix range     [1e+00, 1e+00]
  Objective range  [4e-01, 2e+00]
  Bounds range     [0e+00, 0e+00]
  RHS range        [2e+04, 4e+05]

Presolve removed 1 rows and 0 columns
Presolve time: 0.01s
Presolved: 17 rows, 41 columns, 88 nonzeros

Iteration    Objective       Primal Inf.    Dual Inf.      Time
       0    4.9700000e+05   6.000000e+04   0.000000e+00      0s
      12    7.9900000e+05   0.000000e+00   0.000000e+00      0s

Solved in 12 iterations and 0.01 seconds (0.00 work units)
Optimal objective  7.990000000e+05


,From,To,Flow
,NewYork,Bronx,130000.0
,NewYork,C1,100000.0
,NewYork,C6,40000.0
,NewJersey,Brooklyn,100000.0
,NewJersey,Queens,170000.0
,NewJersey,StatenIsland,80000.0
,Brooklyn,C2,20000.0
,Brooklyn,C4,70000.0
,Queens,C5,120000.0
,StatenIsland,C3,80000.0


## Detected patterns
- New York to C1 and C6 remain with the same values of 100000 and 40000 tons respectively in the three models. This means that the direct flows from the New York generation center to landfills never change in the three studied models.
- New Jersey - Staten Island - C3 remains with the same value of 80000 tons. This route is fully utilized in the three studied models.
- New Jersey - Brooklyn - C2 and New Jersey - Brooklyn - C4 remain with the same values of 20000 and 70000 tons in the three scenarios.
- When adding the landfill C7, it changes the flow from New Jersey - Brooklyn - C5 to New Jersey - Brooklyn - C7. If we analyze this behaviour with the costs per unit, we will see that it changed because New Jersey - Brooklyn - C7 (0.9) is more economic than New Jersey - Brooklyn - C5 (1.2).
- If we analyse the costs for the C7 possible flows, we got the possible prices per ton ordered ascendingly: 0.9 for New Jersey - Brooklyn - C7, 1.1 for New York - Brooklyn - C7, 1.3 for New Jersey - Queens - C7, 1.5 for New York - Bronx - C7, 1.8 for New York - C7, 1.8 for New York - Queens - C7, 1.9 for New Jersey - C7, 2.3 for New York - StatenIsland - C7, 2.3 for New Jersey - StatenIsland - C7. Considering this, the model will search for transportation in this order, having for the route New Jersey - Brooklyn - C7 only 10000 tons possible due to the limit of Brooklyn’s depot capacity. Then we use the flow New Jersey - Queens - C7 to complete the remaining 50000 tons of demand because the flow New York - Brooklyn - C7 is not possible to use because the Brooklyn’s depot is full.
- When adding the landfill C8, it opens the flow New York- Bronx - C8.If we analyse the costs for the C8 possible flows, we got the possible prices per ton ordered ascendingly: 1.4 for New York - Bronx - C8, 1.5 for New York - C8, 1.7 for New Jersey - C8, 1.8 for New York - StatenIsland - C8, 1.8 for New Jersey - StatenIsland - C8, 2.2 for New Jersey - Queens - C8, 2.5 for New Jersey - Brooklyn - C8, 2.7 for New York - Brooklyn - C8, 2.7 for New York - Queens - C8. Considering this, the model will search for transportation in this order, using the route New York - Bronx - C8 for the 130000 tons of demand.


It can be concluded that the rules of the base network remain the same, changing the flows according to the new data available, however, these changes in the flows are not common, because the base network is stable and will only accept changes if it's strictly necessary. The rules of the network are first searching the least possible price and then checking with the constraints to stop assigning tons when they are met.

## Explanation of results

The results of the model use prioritization ranked by cost of transportation in every possible flow to cover the demand of waste for every landfill. It compares the best flows for every landfill and chooses the best combination possible. Additionally, when a flow uses a depot and it reaches its maximum capacity, the depot will stop being used for this and every other flow, even if it’s next in optimal price. Finally, all the waste that enters the depot from the centers is equal to all the waste that leaves the same depot.

Following the rules, the better prices available for the landfills C1 and C6 are New York - C1 and New York - C6. These flows do not interact with depots and will be used to fill the total demand for C1 and C6. The best flows for the rest of the landfills are New Jersey - Brooklyn - C2, New Jersey - Staten Island - C3, New Jersey - Brooklyn - C4 and New Jersey - Brooklyn - C5. The flow New Jersey - Staten Island - C3 will be used to complete the demand of C3. The flow New Jersey - Brooklyn will be used and the model will decide which quantity is needed for C2, C4 and C5. In this case, the landfills C2 and C4 will be completed with these flows, while the landfill C5 will receive the remaining waste of Brooklyn and complement their demand with the next better flow that is New Jersey - Queens - C5.

If the landfill C7 is added, the model following the rules has found that the complete demand of C5 should be taken by the New Jersey - Queens - C5 flow and use the flow New Jersey - Brooklyn - C7  until the capacity of Brooklyn is covered and the flow New Jersey - Queens - C7 to complete the demand of this landfill. Finally, when adding the landfill C8, the model has found that the complete demand of this landfill should be taken by the flow New York - Bronx - C8. As it can be seen, the model will remain stable and will follow the same rules even when new landfills are added, managing the best values of \$541,000 for the six landfills, \$617,000 for the seven landfills and \$799,000 for the eight landfills.

## Limitations and next steps
This is a deterministic linear model, so the result is only as good as the input costs, capacities, and demand assumptions. A production version should include scenario analysis for demand uncertainty, lane disruptions, variable handling costs, and capacity changes.